# Cardiac Nexus — First ECG Model: MI vs Non-MI

This notebook trains a research baseline using the public PTB-XL 12-lead ECG dataset. The task is to classify recordings with a myocardial infarction (MI) diagnostic label versus recordings without an MI label.

> This is a research prototype, not a clinical diagnostic tool. The non-MI group includes other diagnoses and is not equivalent to healthy patients.

## Experiment plan

1. Download PTB-XL from PhysioNet.
2. Read the dataset labels and create a binary MI/non-MI target.
3. Use the official patient-level PTB-XL folds: folds 1–8 for training, 9 for validation, and 10 for testing.
4. Load the 100 Hz, 10-second signals and standardize each lead.
5. Train a small 1D convolutional neural network.
6. Evaluate using AUROC, average precision, F1, sensitivity, specificity, and a confusion matrix.

In [ ]:
!pip -q install wfdb pandas numpy scikit-learn matplotlib seaborn torch tqdm

In [ ]:
from pathlib import Path
import ast
import random

import numpy as np
import pandas as pd
import wfdb
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
    recall_score,
)

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
DATA_ROOT = Path("/content/data")
PTBXL_DIR = DATA_ROOT / "ptb-xl"
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

In [ ]:
# Download PTB-XL once into the temporary Colab workspace.
PTBXL_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (PTBXL_DIR / "ptbxl_database.csv").exists():
    wfdb.dl_database("ptb-xl", dl_dir=str(PTBXL_DIR), version="1.0.3")

metadata = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", index_col=0)
scp = pd.read_csv(PTBXL_DIR / "scp_statements.csv", index_col=0)

# Keep only diagnostic statements that have a diagnostic superclass.
scp = scp[scp["diagnostic"] == 1]
diagnostic_map = scp["diagnostic_class"].dropna().to_dict()

def diagnostic_classes(raw_codes):
    codes = ast.literal_eval(raw_codes) if isinstance(raw_codes, str) else raw_codes
    return {diagnostic_map[code] for code in codes if code in diagnostic_map}

metadata["diagnostic_classes"] = metadata["scp_codes"].apply(diagnostic_classes)
metadata["label"] = metadata["diagnostic_classes"].apply(lambda classes: int("MI" in classes))

print("Total recordings:", len(metadata))
print(metadata["label"].value_counts().rename({0: "non-MI", 1: "MI"}))

In [ ]:
# PTB-XL provides patient-wise official folds, reducing train/test leakage.
train_df = metadata[metadata["strat_fold"].between(1, 8)].copy()
val_df = metadata[metadata["strat_fold"] == 9].copy()
test_df = metadata[metadata["strat_fold"] == 10].copy()

print("Train:", len(train_df), train_df["label"].mean())
print("Validation:", len(val_df), val_df["label"].mean())
print("Test:", len(test_df), test_df["label"].mean())

def load_ecg(row):
    record_path = PTBXL_DIR / row["filename_lr"]
    signal, _ = wfdb.rdsamp(str(record_path))
    signal = signal.astype(np.float32).T  # [12 leads, 1000 samples]
    mean = signal.mean(axis=1, keepdims=True)
    std = signal.std(axis=1, keepdims=True) + 1e-6
    return (signal - mean) / std

class PTBXLDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return torch.tensor(load_ecg(row)), torch.tensor(row["label"], dtype=torch.float32)

train_loader = DataLoader(PTBXLDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(PTBXLDataset(val_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(PTBXLDataset(test_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

sample_x, sample_y = next(iter(train_loader))
print("Batch shape:", sample_x.shape, "Labels shape:", sample_y.shape)

In [ ]:
class ECG1DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(12, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        return self.classifier(self.features(x).squeeze(-1)).squeeze(-1)

model = ECG1DCNN().to(DEVICE)
positive = train_df["label"].sum()
negative = len(train_df) - positive
pos_weight = torch.tensor([negative / max(positive, 1)], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model)
print("Positive-class weight:", pos_weight.item())

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    losses, labels, probabilities = [], [], []
    for signals, targets in tqdm(loader, leave=False):
        signals, targets = signals.to(DEVICE), targets.to(DEVICE)
        with torch.set_grad_enabled(training):
            logits = model(signals)
            loss = criterion(logits, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        losses.append(loss.item() * len(targets))
        labels.extend(targets.detach().cpu().numpy())
        probabilities.extend(torch.sigmoid(logits).detach().cpu().numpy())
    return np.sum(losses) / len(loader.dataset), np.array(labels), np.array(probabilities)

history = []
for epoch in range(1, EPOCHS + 1):
    train_loss, _, _ = run_epoch(train_loader, training=True)
    val_loss, val_y, val_p = run_epoch(val_loader, training=False)
    val_auc = roc_auc_score(val_y, val_p)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_auc": val_auc})
    print(f"Epoch {epoch:02d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_AUROC={val_auc:.4f}")

In [ ]:
history_df = pd.DataFrame(history)
test_loss, test_y, test_p = run_epoch(test_loader, training=False)
test_pred = (test_p >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(test_y, test_pred, labels=[0, 1]).ravel()

metrics = {
    "test_loss": test_loss,
    "AUROC": roc_auc_score(test_y, test_p),
    "Average precision": average_precision_score(test_y, test_p),
    "F1": f1_score(test_y, test_pred, zero_division=0),
    "Sensitivity": tp / max(tp + fn, 1),
    "Specificity": tn / max(tn + fp, 1),
}
print(pd.Series(metrics))
print(classification_report(test_y, test_pred, target_names=["non-MI", "MI"], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(confusion_matrix(test_y, test_pred), annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False)
axes[0].set_title("Test confusion matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
fpr, tpr, _ = roc_curve(test_y, test_p)
axes[1].plot(fpr, tpr, label=f"AUROC = {metrics['AUROC']:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("Test ROC curve")
axes[1].set_xlabel("False-positive rate")
axes[1].set_ylabel("True-positive rate")
axes[1].legend()
plt.tight_layout()

torch.save({"model_state_dict": model.state_dict(), "metrics": metrics}, "/content/cardio_nexus_ecg_mi_baseline.pt")
print("Saved research checkpoint to /content/cardio_nexus_ecg_mi_baseline.pt")

## Interpretation

The test metrics describe performance on the held-out PTB-XL test fold only. They do not establish clinical safety, generalization to Indian patients, or readiness for diagnosis. Record the dataset version, target definition, random seed, metrics, and limitations in the research documentation before comparing future models.